# 🔬 01. Single-Cell RAW FASTQ Quality Control & Exploration
### Single-Cell RNA-Seq (10x Genomics 3' Chromium)
This notebook performs quality control inspection of RAW single-cell sequencing reads prior to preprocessing.

#### Key Objectives:
1. Understand 10x Chromium read architecture (R1: 16bp Cell Barcode + 12bp UMI; R2: cDNA).
2. Evaluate per-cycle Phred quality scores across Cell Barcode, UMI, and cDNA.
3. Compute Cell Barcode diversity and render the **Knee Plot** (Rank vs Read Depth).
4. Measure Whitelist match rates and identify error-correctable barcode populations.


In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

from src.utils import load_config
from src.sc_qc import SingleCellQC

# Load configuration
config = load_config("../config/pipeline_config.yaml")
paths = config["paths"]
print(f"Loaded config for project: {config['project']['name']}")


## 1. Run Single-Cell QC Analysis
Let's initialize the `SingleCellQC` engine on the raw FASTQ files.


In [ ]:
qc_raw = SingleCellQC(
    r1_path=f"../{paths['sample_r1']}",
    r2_path=f"../{paths['sample_r2']}",
    cb_len=config['chemistry']['r1_structure']['cell_barcode_len'],
    umi_len=config['chemistry']['r1_structure']['umi_len'],
    whitelist_path=f"../{paths['whitelist_file']}",
    sample_id="raw_sample_01"
)

raw_results = qc_raw.analyze_fastq()
metrics = raw_results["summary_metrics"]
pd.DataFrame([metrics]).T.rename(columns={0: "Raw Value"})


## 2. Per-Cycle Sequencing Quality Profile
In single-cell 10x sequencing:
- **Cycles 1-16 (R1):** Cell Barcode
- **Cycles 17-28 (R1):** Unique Molecular Identifier (UMI)
- **Cycles 1-91 (R2):** cDNA Transcript read


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# R1 Profile
r1_quals = raw_results["r1_mean_qual_per_cycle"]
ax1.plot(range(1, len(r1_quals) + 1), r1_quals, color="#1f77b4", lw=2, marker='o', markersize=4)
ax1.axvspan(1, 16, color="#4CAF50", alpha=0.15, label="Cell Barcode (1-16bp)")
ax1.axvspan(16, 28, color="#FF9800", alpha=0.15, label="UMI (17-28bp)")
ax1.axhline(30, color="red", linestyle="--", alpha=0.7, label="Q30 Threshold")
ax1.set_title("R1: Cell Barcode & UMI Quality", fontweight='bold')
ax1.set_xlabel("Sequencing Cycle (bp)")
ax1.set_ylabel("Mean Phred Quality Score")
ax1.set_ylim(0, 42)
ax1.legend(loc="lower right")

# R2 Profile
r2_quals = raw_results["r2_mean_qual_per_cycle"]
ax2.plot(range(1, len(r2_quals) + 1), r2_quals, color="#e91e63", lw=2)
ax2.axhline(30, color="red", linestyle="--", alpha=0.7, label="Q30 Threshold")
ax2.set_title("R2: cDNA Quality (3' Degradation)", fontweight='bold')
ax2.set_xlabel("Sequencing Cycle (bp)")
ax2.set_ylabel("Mean Phred Quality Score")
ax2.set_ylim(0, 42)
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()


## 3. Cell Barcode Knee Plot (Barcode Rank vs Read Depth)
The Knee Plot is the foundational QC plot in single-cell genomics:
- High-depth plateau represents **authentic single cells**.
- The steep drop (inflection / knee point) separates captured single cells from **ambient background RNA droplets**.


In [ ]:
counts = np.array(raw_results["barcode_rank_counts"])
ranks = np.arange(1, len(counts) + 1)
knee_stats = raw_results["knee_stats"]
est_cells = knee_stats["estimated_cells"]

plt.figure(figsize=(8, 5.5))
plt.loglog(ranks, counts, color="#3f51b5", lw=2.5, label="Barcode Depth Curve")
plt.axvline(est_cells, color="#e53935", linestyle="--", lw=2, 
            label=f"Knee Cutoff: ~{est_cells} cells ({knee_stats['fraction_reads_in_cells']*100:.1f}% reads)")
plt.scatter([est_cells], [counts[min(est_cells-1, len(counts)-1)]], color="red", s=80, zorder=5)

plt.title("Single-Cell Barcode Rank vs. UMI/Read Count (Knee Plot)", fontweight='bold')
plt.xlabel("Barcode Rank (log10)")
plt.ylabel("Read Depth per Barcode (log10)")
plt.legend(frameon=True, facecolor="white", loc="lower left")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Barcode Whitelist Match Rate Breakdown
Let's see what percentage of raw cell barcodes match the official whitelist exactly, require 1-bp error correction, or are invalid chimeric reads.


In [ ]:
wl_labels = ["Exact Whitelist Match", "1-bp Mismatch (Recoverable)", "Invalid / Noise"]
wl_values = [
    metrics.get("whitelist_exact_match_rate", 0) * 100,
    metrics.get("whitelist_1bp_mismatch_rate", 0) * 100,
    metrics.get("whitelist_invalid_rate", 0) * 100
]
colors = ["#4CAF50", "#FF9800", "#F44336"]

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    wl_values, labels=wl_labels, autopct='%1.2f%%',
    colors=colors, startangle=140, explode=(0, 0.05, 0.1),
    wedgeprops=dict(width=0.6, edgecolor='w')
)
plt.setp(autotexts, size=10, weight="bold")
plt.title("Raw Cell Barcode Whitelist Conformity", fontweight='bold')
plt.tight_layout()
plt.show()
